# M2 학습곡선·용량 탐색 (교수님 언더피팅 지적 대응)
**300 epoch까지 학습하며 25 epoch마다 개발분할을 평가**합니다. 지금까지는 100 epoch 끝에서만 재서, 그 시점이 이른 것인지 늦은 것인지 알 수 없었습니다. 한 번 돌리면 곡선 전체가 나오므로 epoch은 격자에 넣지 않습니다.

조건은 기존 설정에서 **한 번에 하나씩만** 바꿉니다.

| 조건 | 바뀌는 것 |
|---|---|
| `baseline` | 64차원 · L2 1e-3 · rho 0.05 (기존 프로토콜) |
| `wide` | 차원 **128** |
| `light_l2` | L2 **1e-4** |
| `strong_signal` | rho **0.10** |

**공유 설정(차원·L2)이 바뀌면 M1도 같은 설정으로 다시 학습**합니다. M1 3회 + M2 4회 = 시드당 7회, 약 14시간입니다. 세션이 끊기면 다시 실행하세요 — epoch 단위로 재개하고 끝난 조건은 저장된 곡선을 재사용합니다.

여기서는 **epoch도 조건도 고르지 않습니다.** 시드 42는 후보를 추리는 용도이고, 추려진 조건만 5시드로 확인합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'b9d3f647b7115080db53547188ccefe10c9f6724'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
import torch
import lightgcn_clv_m2_capacity_search as search

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = search.configure_capacity_search()
summary = search.preflight_summary(cfg)
assert summary['protocol_epoch'] in summary['evaluated_at_epochs']
assert summary['m1_retrained_per_shared_setting'] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
curve = search.run_capacity_search(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy(); view.attrs = {}; display(view)

print('1) 조건별 학습곡선 (25 epoch 간격)')
show(curve)
print('2) 같은 조건·같은 시드에서 M2 - M1')
show(curve.attrs['gap'])
print('3) 판독')
print(json.dumps(curve.attrs['reading'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(curve.attrs['result_paths'], ensure_ascii=False, indent=2))


In [ ]:
# 교수님 보고용 그림: 개발 성능이 100 epoch 이후에도 오르는가
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, part in curve.groupby('condition'):
    for model_id, line in part.groupby('model_id'):
        axes[0].plot(line.epoch, line['recall@10'], marker='o', markersize=3,
                     label=f'{name} / {model_id.split("_")[0]}')
for name, part in curve.attrs['gap'].groupby('condition'):
    axes[1].plot(part.epoch, part['recall@10'], marker='o', markersize=3, label=name)
for ax, title in zip(axes, ['개발 Recall@10', 'M2 - M1 (Recall@10)']):
    ax.axvline(100, color='gray', linestyle='--', linewidth=1)
    ax.set_xlabel('epoch'); ax.set_title(title); ax.legend(fontsize=7)
axes[1].axhline(0, color='black', linewidth=0.8)
plt.tight_layout(); plt.show()
